# Data cleaning

Builds the tables for each figure from `dataset/ipeds_admissions_panel.csv`. Figure specs are in `CLAUDE.md`.
Run the cells top to bottom. Check the output of step 2 before running step 5.

In [ ]:
from pathlib import Path
import pandas as pd

Path("clean").mkdir(exist_ok=True)
Path("figures").mkdir(exist_ok=True)

## 1. Load
Only the columns we need.

In [ ]:
cols = ["unitid", "cohort_year", "institution_name", "state", "control_label",
        "applicants", "admitted", "enrolled", "sat_submit_pct", "act_submit_pct",
        "act_composite_p25", "act_composite_p50", "act_composite_p75",
        "test_policy_label"]
panel = pd.read_csv("dataset/ipeds_admissions_panel.csv", usecols=cols)
panel.shape

## 2. Look before changing anything
Check the exact `test_policy_label` strings, since step 5 depends on them.

In [ ]:
assert not panel.duplicated(["unitid", "cohort_year"]).any()
panel.groupby("cohort_year")["unitid"].nunique()

In [ ]:
panel["control_label"].value_counts()

In [ ]:
panel["test_policy_label"].value_counts(dropna=False)

In [ ]:
panel[["applicants", "admitted", "enrolled"]].isna().mean()

## 3. States and regions
Keep the 50 states and DC (drops territories) and add Census region.

In [ ]:
regions = {
    "Northeast": "CT ME MA NH RI VT NJ NY PA",
    "Midwest": "IL IN MI OH WI IA KS MN MO NE ND SD",
    "South": "DE DC FL GA MD NC SC VA WV AL KY MS TN AR LA OK TX",
    "West": "AZ CO ID MT NV NM UT WY AK CA HI OR WA",
}
state_to_region = {s: r for r, v in regions.items() for s in v.split()}
panel["region"] = panel["state"].map(state_to_region)
panel = panel.dropna(subset=["region"])
panel["region"].value_counts()

## 4. Bad rows and rates
Drop impossible rows and recompute the rates so they match the counts we sum later. Blanks stay blank, never filled with zero.

In [ ]:
bad = ((panel.admitted > panel.applicants) | (panel.enrolled > panel.admitted)
       | (panel.applicants == 0) | (panel.admitted == 0))
print(bad.sum(), "rows dropped")
panel = panel[~bad].copy()
panel["admit_rate"] = panel["admitted"] / panel["applicants"]
panel["yield_rate"] = panel["enrolled"] / panel["admitted"]

## 5. Groups
Selectivity tier from 2019 admit rate, test policy group from 2019 and 2022 policy, and a score submission measure.

In [ ]:
# selectivity tier from 2019 admit rate
base = panel[panel.cohort_year == 2019].set_index("unitid")["admit_rate"]
tier = pd.cut(base, [0, .25, .5, .75, 1], include_lowest=True,
              labels=["Under 25%", "25-50%", "50-75%", "Over 75%"])
panel["tier"] = panel["unitid"].map(tier)

In [ ]:
# test policy group: required in 2019, then optional or still required in 2022
pol = panel.set_index(["unitid", "cohort_year"])["test_policy_label"].unstack()
req19 = pol[2019] == "Required"
group = pd.Series(pd.NA, index=pol.index, dtype="object")
group[req19 & (pol[2022] == "Required")] = "Stayed required"
group[req19 & pol[2022].notna() & (pol[2022] != "Required")] = "Went optional"
panel["policy_group"] = panel["unitid"].map(group)
group.value_counts(dropna=False)

In [ ]:
# SAT and ACT takers overlap, so the max is a lower bound on "submitted any score"
panel["test_submit_pct"] = panel[["sat_submit_pct", "act_submit_pct"]].max(axis=1)

## 6. Fixed panels and tables
Each figure uses only schools with data in every year. Rates are sums of counts divided, not averages of school rates.

In [ ]:
def fixed_panel(df, cols, n_years=11):
    d = df.dropna(subset=cols)
    n = d.groupby("unitid")["cohort_year"].nunique()
    return d[d["unitid"].isin(n[n == n_years].index)]


def index_to(df, by, value, base):
    t = df.groupby([by, "cohort_year"])[value].sum().unstack("cohort_year")
    return t.div(t[base], axis=0) * 100


def rates(df, by):
    a = df.groupby([by, "cohort_year"], observed=True)[["applicants", "admitted", "enrolled"]].sum()
    a["admit_rate"] = a["admitted"] / a["applicants"]
    a["yield_rate"] = a["enrolled"] / a["admitted"]
    return a

Figures 1 and 2

In [ ]:
enr = fixed_panel(panel, ["enrolled"])
st = enr.groupby(["state", "cohort_year"])["enrolled"].sum().unstack()
map_df = pd.DataFrame({
    "2014-2019": (st[2019] / st[2014] - 1) * 100,
    "2019-2024": (st[2024] / st[2019] - 1) * 100,
    "n_schools": enr.groupby("state")["unitid"].nunique(),
})
idx_control = index_to(enr, "control_label", "enrolled", 2014)
idx_region = index_to(enr, "region", "enrolled", 2014)
map_df.head()

Figures 3 and 5

In [ ]:
cov = fixed_panel(panel, ["applicants", "admitted", "enrolled"])
by_tier = rates(cov, "tier")
by_policy = rates(cov.dropna(subset=["policy_group"]), "policy_group")
by_tier

Figure 4

In [ ]:
tst = fixed_panel(panel.dropna(subset=["policy_group"]), ["test_submit_pct"])
submit = tst.groupby(["policy_group", "cohort_year"])["test_submit_pct"].median().unstack()
submit

Figure 6

In [ ]:
act = fixed_panel(panel.dropna(subset=["policy_group"]), ["act_composite_p25", "act_composite_p75"])
act_med = act.groupby(["policy_group", "cohort_year"])[["act_composite_p25", "act_composite_p75"]].median()
act_med

## 7. Check sizes and save
If a tier or group has only a handful of schools, merge tiers or say so in the caption.

In [ ]:
for name, d in [("enr", enr), ("cov", cov), ("tst", tst), ("act", act)]:
    print(name, d["unitid"].nunique())
print(cov.drop_duplicates("unitid")["tier"].value_counts())
print(cov.drop_duplicates("unitid")["policy_group"].value_counts())
print(map_df["n_schools"].describe())

In [ ]:
map_df.to_csv("clean/map_df.csv")
for name, t in [("idx_control", idx_control), ("idx_region", idx_region), ("by_tier", by_tier),
                ("by_policy", by_policy), ("submit", submit), ("act_med", act_med)]:
    t.to_csv(f"clean/{name}.csv")